<a href="https://colab.research.google.com/github/Kishoby/Conceptual-Research_Hybrid-Approach/blob/Humidity/Humidity_Without_Iterations.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
import pandas as pd
import numpy as np

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import SGDRegressor
from sklearn.ensemble import RandomForestRegressor

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Flatten, Dense, Input

# =========================
# 1. Load datasets
# =========================
train_df = pd.read_csv("/content/drive/MyDrive/Research v2/Research 18.03.2026/Humidity/humidity_training_dataset (1).csv")
test_df = pd.read_csv("/content/drive/MyDrive/Research v2/Research 18.03.2026/Humidity/humidity_testing_dataset (1).csv")

# =========================
# 2. Features & Target
# =========================
features = [
    "uv_index",
    "cloud",
    "condition_text",
    "air_quality_Ozone",
    "temperature_celsius",
    "feels_like_celsius",
    "air_quality_us-epa-index",
    "air_quality_gb-defra-index",
    "longitude",
    "precip_mm",
    "air_quality_PM10",
    "air_quality_PM2.5",
    "visibility_km",
    "air_quality_Sulphur_dioxide",
    "latitude"
]

target = "humidity"

# =========================
# 3. Keep needed columns
# =========================
train_df = train_df[features + [target]].copy()
test_df = test_df[features + [target]].copy()

# condition_text numeric
if "condition_text" in train_df.columns:
    train_df["condition_text"] = pd.to_numeric(train_df["condition_text"], errors="coerce")
    test_df["condition_text"] = pd.to_numeric(test_df["condition_text"], errors="coerce")

# convert all columns to numeric
for col in features + [target]:
    train_df[col] = pd.to_numeric(train_df[col], errors="coerce")
    test_df[col] = pd.to_numeric(test_df[col], errors="coerce")

# drop missing values
train_df = train_df.dropna().reset_index(drop=True)
test_df = test_df.dropna().reset_index(drop=True)

# X and y
X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

print("X_train shape:", X_train.shape)
print("X_test shape :", X_test.shape)

# =========================
# 4. Evaluation function
# =========================
def evaluate_model(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    y_true_safe = np.where(np.array(y_true) == 0, 1e-10, y_true)
    accuracy = 100 - (np.mean(np.abs((y_true - y_pred) / y_true_safe)) * 100)

    return mse, rmse, mae, r2, accuracy

X_train shape: (104002, 15)
X_test shape : (26001, 15)


In [5]:
scaler_sgd = StandardScaler()
X_train_sgd = scaler_sgd.fit_transform(X_train)
X_test_sgd = scaler_sgd.transform(X_test)

sgd_model = SGDRegressor(max_iter=1000, tol=1e-3, random_state=42)
sgd_model.fit(X_train_sgd, y_train)

y_train_pred_sgd = sgd_model.predict(X_train_sgd)
y_test_pred_sgd = sgd_model.predict(X_test_sgd)

train_metrics_sgd = evaluate_model(y_train, y_train_pred_sgd)
test_metrics_sgd = evaluate_model(y_test, y_test_pred_sgd)

print("Linear (SGD) - Training Results")
print("MSE:", train_metrics_sgd[0])
print("RMSE:", train_metrics_sgd[1])
print("MAE:", train_metrics_sgd[2])
print("R2:", train_metrics_sgd[3])
print("Accuracy (%):", train_metrics_sgd[4])

print("\nLinear (SGD) - Testing Results")
print("MSE:", test_metrics_sgd[0])
print("RMSE:", test_metrics_sgd[1])
print("MAE:", test_metrics_sgd[2])
print("R2:", test_metrics_sgd[3])
print("Accuracy (%):", test_metrics_sgd[4])

Linear (SGD) - Training Results
MSE: 353.5713215114916
RMSE: 18.80349226903055
MAE: 12.24318966272126
R2: 0.39385774130468765
Accuracy (%): 71.37336237688962

Linear (SGD) - Testing Results
MSE: 286.39909419634193
RMSE: 16.92332987908532
MAE: 13.142671226018784
R2: 0.40256848999577177
Accuracy (%): 72.70707778777098


In [6]:
scaler_cnn = StandardScaler()
X_train_cnn = scaler_cnn.fit_transform(X_train)
X_test_cnn = scaler_cnn.transform(X_test)

X_train_cnn = X_train_cnn.reshape((X_train_cnn.shape[0], X_train_cnn.shape[1], 1))
X_test_cnn = X_test_cnn.reshape((X_test_cnn.shape[0], X_test_cnn.shape[1], 1))

cnn_model = Sequential([
    Input(shape=(X_train_cnn.shape[1], 1)),
    Conv1D(filters=32, kernel_size=2, activation='relu'),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1)
])

cnn_model.compile(optimizer='adam', loss='mse')

cnn_model.fit(
    X_train_cnn,
    y_train,
    epochs=20,
    batch_size=32,
    verbose=0
)

y_train_pred_cnn = cnn_model.predict(X_train_cnn).flatten()
y_test_pred_cnn = cnn_model.predict(X_test_cnn).flatten()

train_metrics_cnn = evaluate_model(y_train, y_train_pred_cnn)
test_metrics_cnn = evaluate_model(y_test, y_test_pred_cnn)

print("CNN - Training Results")
print("MSE:", train_metrics_cnn[0])
print("RMSE:", train_metrics_cnn[1])
print("MAE:", train_metrics_cnn[2])
print("R2:", train_metrics_cnn[3])
print("Accuracy (%):", train_metrics_cnn[4])

print("\nCNN - Testing Results")
print("MSE:", test_metrics_cnn[0])
print("RMSE:", test_metrics_cnn[1])
print("MAE:", test_metrics_cnn[2])
print("R2:", test_metrics_cnn[3])
print("Accuracy (%):", test_metrics_cnn[4])

3251/3251 ━━━━━━━━━━━━━━━━━━━━ 4s 1ms/step
813/813 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
CNN - Training Results
MSE: 95.17295837402344
RMSE: 9.755662887473276
MAE: 7.208673477172852
R2: 0.8368409872055054
Accuracy (%): 83.9881029351155

CNN - Testing Results
MSE: 171.25161743164062
RMSE: 13.086314127042826
MAE: 9.59539794921875
R2: 0.6427673101425171
Accuracy (%): 80.26077053864549


In [7]:
rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

rf_model.fit(X_train, y_train)

y_train_pred_rf = rf_model.predict(X_train)
y_test_pred_rf = rf_model.predict(X_test)

train_metrics_rf = evaluate_model(y_train, y_train_pred_rf)
test_metrics_rf = evaluate_model(y_test, y_test_pred_rf)

print("Random Forest - Training Results")
print("MSE:", train_metrics_rf[0])
print("RMSE:", train_metrics_rf[1])
print("MAE:", train_metrics_rf[2])
print("R2:", train_metrics_rf[3])
print("Accuracy (%):", train_metrics_rf[4])

print("\nRandom Forest - Testing Results")
print("MSE:", test_metrics_rf[0])
print("RMSE:", test_metrics_rf[1])
print("MAE:", test_metrics_rf[2])
print("R2:", test_metrics_rf[3])
print("Accuracy (%):", test_metrics_rf[4])

Random Forest - Training Results
MSE: 8.977567541970346
RMSE: 2.9962589243872677
MAE: 2.104388280994596
R2: 0.9846093765630741
Accuracy (%): 95.48221112525962

Random Forest - Testing Results
MSE: 174.04910929195032
RMSE: 13.192767309853924
MAE: 9.03876427829699
R2: 0.6369317351684943
Accuracy (%): 80.66643246667347


In [8]:
xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=6,
    random_state=42
)

xgb_model.fit(X_train, y_train)

y_train_pred_xgb = xgb_model.predict(X_train)
y_test_pred_xgb = xgb_model.predict(X_test)

train_metrics_xgb = evaluate_model(y_train, y_train_pred_xgb)
test_metrics_xgb = evaluate_model(y_test, y_test_pred_xgb)

print("XGBoost - Training Results")
print("MSE:", train_metrics_xgb[0])
print("RMSE:", train_metrics_xgb[1])
print("MAE:", train_metrics_xgb[2])
print("R2:", train_metrics_xgb[3])
print("Accuracy (%):", train_metrics_xgb[4])

print("\nXGBoost - Testing Results")
print("MSE:", test_metrics_xgb[0])
print("RMSE:", test_metrics_xgb[1])
print("MAE:", test_metrics_xgb[2])
print("R2:", test_metrics_xgb[3])
print("Accuracy (%):", test_metrics_xgb[4])

XGBoost - Training Results
MSE: 73.2159652709961
RMSE: 8.556632823196056
MAE: 6.352118015289307
R2: 0.8744827508926392
Accuracy (%): 86.11113616897777

XGBoost - Testing Results
MSE: 167.89892578125
RMSE: 12.957581787557816
MAE: 9.194829940795898
R2: 0.6497610807418823
Accuracy (%): 80.64371133591878


In [9]:
lgbm_model = LGBMRegressor(
    n_estimators=100,
    learning_rate=0.1,
    random_state=42
)

lgbm_model.fit(X_train, y_train)

y_train_pred_lgbm = lgbm_model.predict(X_train)
y_test_pred_lgbm = lgbm_model.predict(X_test)

train_metrics_lgbm = evaluate_model(y_train, y_train_pred_lgbm)
test_metrics_lgbm = evaluate_model(y_test, y_test_pred_lgbm)

print("LightGBM - Training Results")
print("MSE:", train_metrics_lgbm[0])
print("RMSE:", train_metrics_lgbm[1])
print("MAE:", train_metrics_lgbm[2])
print("R2:", train_metrics_lgbm[3])
print("Accuracy (%):", train_metrics_lgbm[4])

print("\nLightGBM - Testing Results")
print("MSE:", test_metrics_lgbm[0])
print("RMSE:", test_metrics_lgbm[1])
print("MAE:", test_metrics_lgbm[2])
print("R2:", test_metrics_lgbm[3])
print("Accuracy (%):", test_metrics_lgbm[4])

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.033375 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2685
[LightGBM] [Info] Number of data points in the train set: 104002, number of used features: 15
[LightGBM] [Info] Start training from score 64.603508
LightGBM - Training Results
MSE: 76.82430256403056
RMSE: 8.764947379421656
MAE: 6.538911377773225
R2: 0.8682968514533776
Accuracy (%): 85.69016027677569

LightGBM - Testing Results
MSE: 166.77071926117907
RMSE: 12.91397379822257
MAE: 9.190143256421042
R2: 0.6521145330006075
Accuracy (%): 80.45234457432737


In [10]:
# =========================
# Final Result Tables Only
# =========================

training_results_table = pd.DataFrame([
    {
        "Model": "Linear (SGD)",
        "MSE": train_metrics_sgd[0],
        "RMSE": train_metrics_sgd[1],
        "MAE": train_metrics_sgd[2],
        "R2": train_metrics_sgd[3],
        "Accuracy (%)": train_metrics_sgd[4]
    },
    {
        "Model": "CNN",
        "MSE": train_metrics_cnn[0],
        "RMSE": train_metrics_cnn[1],
        "MAE": train_metrics_cnn[2],
        "R2": train_metrics_cnn[3],
        "Accuracy (%)": train_metrics_cnn[4]
    },
    {
        "Model": "Random Forest",
        "MSE": train_metrics_rf[0],
        "RMSE": train_metrics_rf[1],
        "MAE": train_metrics_rf[2],
        "R2": train_metrics_rf[3],
        "Accuracy (%)": train_metrics_rf[4]
    },
    {
        "Model": "XGBoost",
        "MSE": train_metrics_xgb[0],
        "RMSE": train_metrics_xgb[1],
        "MAE": train_metrics_xgb[2],
        "R2": train_metrics_xgb[3],
        "Accuracy (%)": train_metrics_xgb[4]
    },
    {
        "Model": "LightGBM",
        "MSE": train_metrics_lgbm[0],
        "RMSE": train_metrics_lgbm[1],
        "MAE": train_metrics_lgbm[2],
        "R2": train_metrics_lgbm[3],
        "Accuracy (%)": train_metrics_lgbm[4]
    }
]).round(4)

testing_results_table = pd.DataFrame([
    {
        "Model": "Linear (SGD)",
        "MSE": test_metrics_sgd[0],
        "RMSE": test_metrics_sgd[1],
        "MAE": test_metrics_sgd[2],
        "R2": test_metrics_sgd[3],
        "Accuracy (%)": test_metrics_sgd[4]
    },
    {
        "Model": "CNN",
        "MSE": test_metrics_cnn[0],
        "RMSE": test_metrics_cnn[1],
        "MAE": test_metrics_cnn[2],
        "R2": test_metrics_cnn[3],
        "Accuracy (%)": test_metrics_cnn[4]
    },
    {
        "Model": "Random Forest",
        "MSE": test_metrics_rf[0],
        "RMSE": test_metrics_rf[1],
        "MAE": test_metrics_rf[2],
        "R2": test_metrics_rf[3],
        "Accuracy (%)": test_metrics_rf[4]
    },
    {
        "Model": "XGBoost",
        "MSE": test_metrics_xgb[0],
        "RMSE": test_metrics_xgb[1],
        "MAE": test_metrics_xgb[2],
        "R2": test_metrics_xgb[3],
        "Accuracy (%)": test_metrics_xgb[4]
    },
    {
        "Model": "LightGBM",
        "MSE": test_metrics_lgbm[0],
        "RMSE": test_metrics_lgbm[1],
        "MAE": test_metrics_lgbm[2],
        "R2": test_metrics_lgbm[3],
        "Accuracy (%)": test_metrics_lgbm[4]
    }
]).round(4)

print("TRAINING RESULTS TABLE")
print(training_results_table)

print("\nTESTING RESULTS TABLE")
print(testing_results_table)

TRAINING RESULTS TABLE
           Model       MSE     RMSE      MAE      R2  Accuracy (%)
0   Linear (SGD)  353.5713  18.8035  12.2432  0.3939       71.3734
1            CNN   95.1730   9.7557   7.2087  0.8368       83.9881
2  Random Forest    8.9776   2.9963   2.1044  0.9846       95.4822
3        XGBoost   73.2160   8.5566   6.3521  0.8745       86.1111
4       LightGBM   76.8243   8.7649   6.5389  0.8683       85.6902

TESTING RESULTS TABLE
           Model       MSE     RMSE      MAE      R2  Accuracy (%)
0   Linear (SGD)  286.3991  16.9233  13.1427  0.4026       72.7071
1            CNN  171.2516  13.0863   9.5954  0.6428       80.2608
2  Random Forest  174.0491  13.1928   9.0388  0.6369       80.6664
3        XGBoost  167.8989  12.9576   9.1948  0.6498       80.6437
4       LightGBM  166.7707  12.9140   9.1901  0.6521       80.4523


In [11]:
display(training_results_table)
display(testing_results_table)

,Model,MSE,RMSE,MAE,R2,Accuracy (%)
0,Linear (SGD),353.5713,18.8035,12.2432,0.3939,71.3734
1,CNN,95.1730,9.7557,7.2087,0.8368,83.9881
2,Random Forest,8.9776,2.9963,2.1044,0.9846,95.4822
3,XGBoost,73.2160,8.5566,6.3521,0.8745,86.1111
4,LightGBM,76.8243,8.7649,6.5389,0.8683,85.6902


,Model,MSE,RMSE,MAE,R2,Accuracy (%)
0,Linear (SGD),286.3991,16.9233,13.1427,0.4026,72.7071
1,CNN,171.2516,13.0863,9.5954,0.6428,80.2608
2,Random Forest,174.0491,13.1928,9.0388,0.6369,80.6664
3,XGBoost,167.8989,12.9576,9.1948,0.6498,80.6437
4,LightGBM,166.7707,12.9140,9.1901,0.6521,80.4523


In [12]:
# Save both tables
training_results_table.to_csv("training_results_table.csv", index=False)
testing_results_table.to_csv("testing_results_table.csv", index=False)

# Download
from google.colab import files

files.download("training_results_table.csv")
files.download("testing_results_table.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>